# Complexity and Searching

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Predict from the input limits whether an approach fits the time limit
- Replace a nested loop with a hash set, sorting or two pointers
- Find boundaries in sorted data with binary search and count the values in a range
- Binary-search the answer when checking a candidate is easy but finding it directly is hard

## In short

Time limits turn Big O into a budget. A computer does roughly **10⁸ simple steps per second**, so read the input limits before writing code: with n up to 200,000, a loop over all pairs is 4 × 10¹⁰ steps (minutes), while O(n log n) is about 3.5 million (milliseconds). As a rule of thumb, n ≤ 5,000 allows O(n²), and n ≤ 500,000 needs O(n log n) or better.

**Run it and watch the columns:** when n doubles, one loop does twice the work, and a loop in a loop does four times as much.

In [ ]:
for (int n = 1000; n <= 16000; n *= 2) {
    long one = 0;
    long nested = 0;
    for (int i = 0; i < n; i++) {
        one++;
    }
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            nested++;
        }
    }
    IO.println(String.format("n = %,6d   one loop: %,7d   loop in a loop: %,12d", n, one, nested));
}

Most fixes replace a **nested loop** with a better tool:

- **A hash set** answers "have I seen this before?" in constant time on average.
- **Sorting** puts equal or close values side by side, so one pass compares neighbours.
- **Two pointers** walk a sorted array from both ends, or as the start and end of a window, each moving one way only.

**Time both ways** of finding repeated ids. Then set `n` to 40,000: which time doubles, and which grows four times?

In [ ]:
int n = 20_000;
int[] ids = new Random(1).ints(n, 0, 1_000_000_000).toArray();

long start = System.nanoTime();
int repeatsSlow = 0; // ids already seen earlier in the list
for (int i = 0; i < n; i++) {
    for (int j = 0; j < i; j++) {
        if (ids[i] == ids[j]) {
            repeatsSlow++;
            break;
        }
    }
}
long middle = System.nanoTime();
Set<Integer> seen = new HashSet<>();
int repeatsFast = 0;
for (int id : ids) {
    if (!seen.add(id)) {
        repeatsFast++;
    }
}
long end = System.nanoTime();

IO.println("loop in a loop: " + (middle - start) / 1_000_000 + " ms, repeats: " + repeatsSlow);
IO.println("HashSet:        " + (end - middle) / 1_000_000 + " ms, repeats: " + repeatsFast);

**Binary search** does more than find one value. In sorted data it finds **boundaries**, such as the first index whose value is ≥ x, and two boundaries count everything in a range:

![The first value ≥ 4 and the first value > 4 in a sorted array](boundaries.svg)

The next cell starts with `// shared`, so it defines `lowerBound` for every cell below it. Run it, then the one that uses it.

In [ ]:
// shared: the cells below can call lowerBound
/** The first index whose value is >= x, or a.length if there is none. */
int lowerBound(int[] a, long x) {
    int lo = 0;
    int hi = a.length;
    while (lo < hi) {
        int mid = (lo + hi) >>> 1;
        if (a[mid] < x) {
            lo = mid + 1;
        } else {
            hi = mid;
        }
    }
    return lo;
}

In [ ]:
int[] a = {2, 4, 4, 4, 7, 9};
IO.println("first >= 4: index " + lowerBound(a, 4));
IO.println("first > 4:  index " + lowerBound(a, 5)); // "first > 4" is "first >= 5"
IO.println("how many 4s: " + (lowerBound(a, 5) - lowerBound(a, 4)));
IO.println("first >= 10: index " + lowerBound(a, 10) + " (past the end)");
// Try it: how many values are between 3 and 8, inclusive?

It can also search for an **answer**: when "does c work?" is no up to some value and yes from there on, binary search finds that first yes in about 30 checks, even between 1 and 10⁹.

**Binary search on the answer:** the smallest x with x × x ≥ target. Try other targets.

In [ ]:
long target = 2_000_000_000_000L; // any target from 0 to 9_000_000_000_000_000_000L
long lo = 0;
long hi = 3_037_000_499L; // the largest x whose x * x fits in a long
int checks = 0;
while (lo < hi) {
    long mid = (lo + hi) / 2;
    checks++;
    if (mid * mid >= target) {
        hi = mid;
    } else {
        lo = mid + 1;
    }
}
IO.println("smallest x: " + lo + ", found in " + checks + " checks");

The usual bug is being off by one: decide whether `hi` is the last index or one past it, and keep that choice everywhere.

## Readings

- [Time Complexity](https://usaco.guide/bronze/time-comp) (USACO Guide, 15 min): The table under "Common Complexities and Constraints": it turns an input limit into the slowest complexity that still fits. The examples can be shown in Java: pick it in the site's settings.
- [Binary Search](https://usaco.guide/silver/binary-search) (USACO Guide, 20 min): Binary search on a yes/no function (the first x where f(x) is true) and the three common mistakes: off by one, negative bounds and overflow. The first index with a value ≥ x is the same search, with f(i) = "a[i] ≥ x". Skip the problem lists.
- [Two Pointers](https://usaco.guide/silver/two-pointers) (USACO Guide, 12 min): "Sum of Two Values" (pointers from both ends of a sorted array) and "Sliding Window" (a start and an end that only move right, shown in an animation). Skip the problem lists.

## Check yourself

1. With n ≤ 100,000, does an O(n²) solution fit a one-second limit?
2. In `2 4 4 4 7 9`, what is the first index with a value ≥ 5?
3. Why can you binary-search the smallest truck capacity that works, when no array holds the capacities?

## Challenges

Solve the unit's challenges right here: each cell starts with `// sphinx: challenge …`, and running it runs **all** of the challenge's tests, hidden ones included. A pass counts in your progress in the Sphinx sidebar, just like Submit. The full description, with examples and hints, opens from the sidebar.

### Repeated Tickets · Easy

In [ ]:
// sphinx: challenge repeated-tickets
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] parts = IO.readln().trim().split(" +");
    int[] ids = new int[n];
    for (int i = 0; i < n; i++) {
        ids[i] = Integer.parseInt(parts[i]);
    }

    // TODO: count how many different ids appear more than once, fast enough for 200,000 tickets
}

### Log Window · Medium

In [ ]:
// sphinx: challenge log-window
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] parts = IO.readln().trim().split(" +");
    int[] times = new int[n];
    for (int i = 0; i < n; i++) {
        times[i] = Integer.parseInt(parts[i]);
    }
    int q = Integer.parseInt(IO.readln().trim());
    long total = 0;
    int busiest = 0;
    for (int i = 0; i < q; i++) {
        String[] window = IO.readln().trim().split(" +");
        int a = Integer.parseInt(window[0]);
        int b = Integer.parseInt(window[1]);

        // TODO: count the timestamps with a <= t <= b using binary search, and update total and busiest
    }
    IO.println("total: " + total);
    IO.println("busiest: " + busiest);
}

### Truck Capacity · Medium

In [ ]:
// sphinx: challenge truck-capacity
void main() {
    String[] first = IO.readln().trim().split(" +");
    int n = Integer.parseInt(first[0]);
    int d = Integer.parseInt(first[1]);
    String[] parts = IO.readln().trim().split(" +");
    int[] weights = new int[n];
    for (int i = 0; i < n; i++) {
        weights[i] = Integer.parseInt(parts[i]);
    }

    // TODO: binary-search the smallest capacity that ships every package in at most d days
}

### Rate Limiter · Hard

In [ ]:
// sphinx: challenge rate-limiter
void main() {
    String[] first = IO.readln().trim().split(" +");
    int n = Integer.parseInt(first[0]);
    int w = Integer.parseInt(first[1]);
    int limit = Integer.parseInt(first[2]);
    String[] parts = IO.readln().trim().split(" +");
    int[] times = new int[n];
    for (int i = 0; i < n; i++) {
        times[i] = Integer.parseInt(parts[i]);
    }

    // TODO: sort the times, find the busiest window with two pointers, then simulate the limiter
}